In [ ]:
# -------------------------------------------------------------
# Common pre‑amble – shared across all notebooks
# -------------------------------------------------------------
from config.notebook_setup import *


# 06 – Semantic Search Demo

Interactive semantic search on the Reuters documents using a MiniLM embedding model. Optionally, this notebook can switch to a Retrieval‑Augmented Generation (RAG) mode that synthesises an answer from the top retrieved documents.

In [ ]:

import os, pathlib, pickle, faiss
import numpy as np
from sentence_transformers import SentenceTransformer
from src.dataset import load_data

EMB_DIR = pathlib.Path('embeddings')
EMB_DIR.mkdir(exist_ok=True, parents=True)
INDEX_PATH = EMB_DIR / 'minilm.index'
DOCS_PATH  = EMB_DIR / 'docs.pkl'
MODEL_NAME = 'sentence-transformers/all-MiniLM-L6-v2'
N_CLASSES  = 10


In [ ]:

# Load documents (train + test for demo)
X_train, y_train, X_test, y_test, label_names = load_data(N_CLASSES)
docs = X_train + X_test
if not INDEX_PATH.exists():
    print('Building embeddings…')
    model = SentenceTransformer(MODEL_NAME)
    emb = model.encode(docs, show_progress_bar=True, batch_size=64, convert_to_numpy=True)
    dimension = emb.shape[1]
    index = faiss.IndexFlatIP(dimension)
    # normalise for cosine sim
    faiss.normalize_L2(emb)
    index.add(emb)
    faiss.write_index(index, str(INDEX_PATH))
    with open(DOCS_PATH, 'wb') as f:
        pickle.dump(docs, f)
    print(f'Index and docs saved to {EMB_DIR}')
else:
    print('Embeddings already built. Loading…')
    index = faiss.read_index(str(INDEX_PATH))
    with open(DOCS_PATH, 'rb') as f:
        docs = pickle.load(f)
    model = SentenceTransformer(MODEL_NAME)


In [ ]:

# ---- Search helper ----
def search(query: str, k: int = 5):
    q_emb = model.encode([query], convert_to_numpy=True)
    faiss.normalize_L2(q_emb)
    D, I = index.search(q_emb, k)
    print(f"Top {k} results:")
    for rank, (idx, score) in enumerate(zip(I[0], D[0]), 1):
        print(f"{rank}. (score={score:.3f}) {docs[idx][:200]}…\n")


In [ ]:

# Demo
search("oil prices in saudi arabia")


### Optional – Retrieval‑Augmented Generation (RAG)
If you have an OpenAI key configured, you can uncomment the cell below to generate answers from the retrieved passages.

In [ ]:

# !pip install openai
import openai, textwrap
openai.api_key = os.getenv('OPENAI_API_KEY')
if not openai.api_key:
    raise RuntimeError('OPENAI_API_KEY env var not set.')

def rag_answer(question: str, k: int = 5) -> str:
    q_emb = model.encode([question], convert_to_numpy=True)
    faiss.normalize_L2(q_emb)
    D, I = index.search(q_emb, k)
    context = "\n".join([docs[i] for i in I[0]])
    prompt = f"Answer the question based only on the context below.\n\nContext:\n{context}\n\nQuestion: {question}\nAnswer:"
    response = openai.Completion.create(model='gpt-3.5-turbo-instruct', prompt=prompt, max_tokens=256)
    return textwrap.dedent(response['choices'][0]['text']).strip()

# Example
# print(rag_answer("What is the outlook for the US dollar this week?"))
